<a href="https://colab.research.google.com/github/PrasannaMadiwar/LLM-from-Scratch-/blob/main/Modifed_GPT_2(Normalization).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding('gpt2')

In [ ]:
class MultiHead(nn.Module):
  def __init__(self,d_in,d_out,context_length,drop_rate,n_heads,qkv_bias=True):

    super().__init__()
    assert(d_out % n_heads == 0)

    self.d_out = d_out
    self.n_heads = n_heads
    self.head_d = d_out // n_heads

    self.query = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.key = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.value = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.proj = nn.Linear(d_out, d_out)
    self.dropout = nn.Dropout(drop_rate)

    self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))


  def forward(self,x):
    b,n_tokens,d_in = x.shape

    keys = self.key(x)
    values = self.value(x)
    queries = self.query(x)

    keys = keys.view(b,n_tokens,self.n_heads,self.head_d)
    values = values.view(b,n_tokens,self.n_heads,self.head_d)
    queries = queries.view(b,n_tokens,self.n_heads,self.head_d)

    keys = keys.transpose(1,2)
    values = values.transpose(1,2)
    queries = queries.transpose(1,2)

    attn_score = queries @ keys.transpose(2,3)
    mask_bool = self.mask.bool()[:n_tokens,:n_tokens]
    attn_score.masked_fill(mask_bool,-1e4)
    attn_weights = torch.softmax(attn_score / keys.shape[-1]**0.5,dim=-1)
    attn_weights = self.dropout(attn_weights)

    context_vec = (attn_weights @ values).transpose(1,2)
    context_vec = context_vec.contiguous().view(b,n_tokens,self.d_out)
    context_vec = self.proj(context_vec)

    return context_vec


In [ ]:
class LayerNorm(nn.Module):

  def __init__(self,d_in):
    super().__init__()
    self.esp = 1e-5
    self.scale = nn.Parameter(torch.ones(d_in))
    self.shift = nn.Parameter(torch.zeros(d_in))
  def forward(self,x):
    mean = x.mean(dim=-1,keepdim=True)
    var = x.var(dim=-1,keepdim=True,unbiased=False)
    norm_x = (x - mean) / torch.sqrt(var + self.esp)

    return self.scale*norm_x+self.shift


In [ ]:
class Gelu(nn.Module):
  def __init__(self):
    super().__init__()
  def forward(self,x):
    return 0.5 * x * (1 + torch.tanh(x * 0.7978845608 * (1 + 0.044715 * x * x)))

In [ ]:
class FeedForward(nn.Module):
  def __init__(self,cfg):
    super().__init__()

    self.layers = nn.Sequential(
        nn.Linear(cfg['d_in'],cfg['d_in']*4),
        Gelu(),
        nn.Linear(cfg['d_in']*4,cfg['d_in'])
    )
  def forward(self,x):
    return self.layers(x)

In [ ]:
class TransFormer(nn.Module):
  def __init__(self,cfg):
    super().__init__()

    self.norm1 = LayerNorm(cfg['d_in'])
    self.norm1_post = LayerNorm(cfg['d_in'])
    self.norm2 = LayerNorm(cfg['d_in'])
    self.norm2_post = LayerNorm(cfg['d_in'])
    self.M_attn = MultiHead(cfg['d_in'],cfg['d_out'],cfg['context_length'],cfg['drop_rate'],cfg['n_heads'],cfg['qkv_bias'])
    self.ff = FeedForward(cfg)
    self.drop_out = nn.Dropout(cfg['drop_rate'])

  def forward(self,x):
    shortcut = x
    x = self.norm1(x)
    x = self.M_attn(x)
    x = self.drop_out(x)
    x = self.norm1_post(x)
    x = shortcut + x

    shortcut = x
    x = self.norm2(x)
    x = self.ff(x)
    x =  self.drop_out(x)
    x = self.norm2_post(x)
    x = shortcut + x

    return x

In [ ]:
class GPT_modified(nn.Module):
  def __init__(self,cfg):

    super().__init__()

    self.word_emb = nn.Embedding(cfg['vocab_size'],cfg['d_out'])
    self.pos_emb = nn.Embedding(cfg['context_length'],cfg['d_out'])
    self.dropout = nn.Dropout(cfg['drop_rate'])
    self.trf = nn.Sequential(
        *[TransFormer(cfg) for i in range(cfg['n_layers'])]
    )
    self.first_norm = LayerNorm(cfg['d_in'])
    self.final_norm = LayerNorm(cfg['d_out'])
    self.output_head = nn.Linear(cfg['d_out'],cfg['vocab_size'],bias=False)

  def forward(self,ind_x):
    b,n_seq =  ind_x.shape
    tok_emb = self.word_emb(ind_x)
    pos_emb = self.pos_emb(torch.arange(n_seq,device=ind_x.device))
    x = tok_emb + pos_emb
    x = self.dropout(x)
    x = self.first_norm(x)
    x = self.trf(x)
    x = self.final_norm(x)
    logits = self.output_head(x)
    return logits

In [ ]:
cfg = {
    'vocab_size': 50257,
    'd_out': 1024,
    'd_in': 1024,
    'context_length': 1024,
    'drop_rate': 0.0,
    'n_heads': 16,
    'n_layers': 24,
    'qkv_bias': True
}


In [ ]:
model = GPT_modified(cfg)
model.to(device)

GPT_modified(
  (word_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (dropout): Dropout(p=0.0, inplace=False)
  (trf): Sequential(
    (0): TransFormer(
      (norm1): LayerNorm()
      (norm1_post): LayerNorm()
      (norm2): LayerNorm()
      (norm2_post): LayerNorm()
      (M_attn): MultiHead(
        (query): Linear(in_features=1024, out_features=1024, bias=True)
        (key): Linear(in_features=1024, out_features=1024, bias=True)
        (value): Linear(in_features=1024, out_features=1024, bias=True)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): Gelu()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (drop_out): Dropout(p=0.0, inplace=False)
    )
    (1): TransFormer(
      (norm1): LayerNorm()
   

In [ ]:
def encode(text, tokenizer, device):
    ids = tokenizer.encode(text)
    return torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

def decode(ids, tokenizer):
    return tokenizer.decode(ids.squeeze(0).cpu().tolist())

In [ ]:
'''from transformers import GPT2LMHeadModel

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
sd = gpt2.state_dict()'''

'from transformers import GPT2LMHeadModel\n\ngpt2 = GPT2LMHeadModel.from_pretrained("gpt2")\nsd = gpt2.state_dict()'

In [ ]:
from transformers import GPT2LMHeadModel

gpt2_medium = GPT2LMHeadModel.from_pretrained("gpt2-medium")
sd = gpt2_medium.state_dict()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
def copy_weights(model):

  model.word_emb.weight.data.copy_(sd["transformer.wte.weight"])
  model.pos_emb.weight.data.copy_(sd["transformer.wpe.weight"])
  model.first_norm.scale.data.copy_(sd["transformer.ln_f.weight"])
  model.first_norm.shift.data.copy_(sd["transformer.ln_f.bias"])
  model.final_norm.scale.data.copy_(sd["transformer.ln_f.weight"])
  model.final_norm.shift.data.copy_(sd["transformer.ln_f.bias"])
  model.output_head.weight.data.copy_(sd["transformer.wte.weight"])


  for i in range(24):
      block = model.trf[i]
      prefix = f"transformer.h.{i}."

      block.norm1.scale.data.copy_(sd[prefix + "ln_1.weight"])
      block.norm1.shift.data.copy_(sd[prefix + "ln_1.bias"])
      block.norm1_post.scale.data.copy_(sd[prefix + "ln_1.weight"])
      block.norm1_post.shift.data.copy_(sd[prefix + "ln_1.bias"])

      qkv_w = sd[prefix + "attn.c_attn.weight"]
      qkv_b = sd[prefix + "attn.c_attn.bias"]

      qw, kw, vw = qkv_w.split(1024, dim=1) #bhau ithe changes krache ahe according to model size apan select krnr
      qb, kb, vb = qkv_b.split(1024) # ani ithe pn

      block.M_attn.query.weight.data.copy_(qw.T)
      block.M_attn.query.bias.data.copy_(qb.T)
      block.M_attn.key.weight.data.copy_(kw.T)
      block.M_attn.key.bias.data.copy_(kb.T)
      block.M_attn.value.weight.data.copy_(vw.T)
      block.M_attn.value.bias.data.copy_(vb.T)
      block.M_attn.proj.weight.data.copy_(sd[prefix + "attn.c_proj.weight"].T)
      block.M_attn.proj.bias.data.copy_(sd[prefix + "attn.c_proj.bias"])

      block.ff.layers[0].weight.data.copy_(sd[prefix + "mlp.c_fc.weight"].T)
      block.ff.layers[0].bias.data.copy_(sd[prefix + "mlp.c_fc.bias"].T)
      block.ff.layers[2].weight.data.copy_(sd[prefix + "mlp.c_proj.weight"].T)
      block.ff.layers[2].bias.data.copy_(sd[prefix + "mlp.c_proj.bias"].T)

      block.norm2.scale.data.copy_(sd[prefix + "ln_2.weight"])
      block.norm2.shift.data.copy_(sd[prefix + "ln_2.bias"])
      block.norm2_post.scale.data.copy_(sd[prefix + "ln_2.weight"])
      block.norm2_post.shift.data.copy_(sd[prefix + "ln_2.bias"])

In [ ]:
copy_weights(model=model)

/tmp/ipython-input-4076984611.py:28: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4416.)
  block.M_attn.query.bias.data.copy_(qb.T)


In [ ]:
def evaluate_model(model,train_loader,val_loader,device,eval_iter):
  model.eval()
  with torch.no_grad():
    val_loss = calc_loss_loader(val_loader,model,device,num_batches=eval_iter)
    train_loss = calc_loss_loader(train_loader,model,device,num_batches=eval_iter)
    model.train()
  return train_loss,val_loss

def calc_loss_batch(input_batch,target_batch,model,device):
  input_batch,target_batch = input_batch.to(device),target_batch.to(device)
  logits = model(input_batch)
  loss = torch.nn.functional.cross_entropy(logits.flatten(0,1),target_batch.flatten(),ignore_index=-100)
  return loss

def calc_loss_loader(data_loader,model,device,num_batches=None):
  total_loss = 0
  if len(data_loader)<0:
    return float('nan')
  elif num_batches is None:
    num_batches = len(data_loader)
  else:
    num_batches= min(num_batches,len(data_loader))
  for i, (inpuut_batch,target_batch) in enumerate(data_loader):
    if i < num_batches:
      loss = calc_loss_batch(input_batch=inpuut_batch,target_batch=target_batch,model=model,device=device)
      total_loss += loss.item()
    else:
      break
  return total_loss/num_batches

In [ ]:
def trainGpt(model,train_loader,val_loader,optimizer,device,num_epoch,eval_iter,eval_freq,tokenizer):
  train_losses,val_losses,seen_tokens = [],[],[]
  token_seen,global_step = 0,-1

  for i in range(num_epoch):
    model.train()

    for input_batch,target_batch in train_loader:
      optimizer.zero_grad()
      loss = calc_loss_batch(input_batch,target_batch,model,device)
      loss.backward()
      optimizer.step()
      train_losses.append(loss.item())
      token_seen += input_batch.numel()
      global_step += 1

      if global_step % eval_freq == 0:
        train_loss,val_loss = evaluate_model(model,train_loader,val_loader,device,eval_iter)
        print("Epoch:"+str(i)+" Training_Loss:"+str(train_loss)+" Validation_Loss:"+str(val_loss))
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        seen_tokens.append(token_seen)

  return train_losses,val_losses,seen_tokens

In [ ]:
import json
import os
import urllib
import ssl

def download_and_load_file(file_path, url):
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url, context=ssl_context) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


In [ ]:
data[0]

{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}

In [ ]:
train_end = int(0.85*len(data))
val_end = int(0.1*len(data)) + train_end

In [ ]:
train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

In [ ]:
len(train_data),len(val_data),len(test_data)

(935, 110, 55)

In [ ]:
def formate_input(entry):
  instrction_text = (
      f"Question:\n{entry['instruction']}"
  )
  input_text = f"\n{entry['input']}" if entry['input'] else ""
  return instrction_text + input_text

In [ ]:
def formate_input1(entry):

  input_text = (
      f"<|system|>"
      f"\n {entry['instruction']}\n\n"
      f"<|user|>"
      f"\n {entry['input']}"
  )
  return input_text

In [ ]:
class IstructionDataset(Dataset):
  def __init__(self,data,tokenizer):
    self.data = data
    self.encode = []

    for entry in self.data:
      input_text = formate_input(entry)
      response_text = f"Answer: \n{entry['output']}"
      text = input_text + response_text
      self.encode.append(tokenizer.encode(text))

  def __len__(self):
    return len(self.encode)
  def __getitem__(self,index):
    return self.encode[index]

In [ ]:
def custom_collate_fn(batch, pad_token_id=50256,ignore_index=-100,allowed_max_length=None,device=device):

    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:

        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [ ]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]], device='cuda:0')
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]], device='cuda:0')


In [ ]:
from functools import partial

In [ ]:
collate_fn = partial(custom_collate_fn,allowed_max_length=1024,device=device)

In [ ]:
train_dataset = IstructionDataset(train_data,tokenizer=tokenizer)
val_dataset = IstructionDataset(val_data,tokenizer=tokenizer)
test_dataset = IstructionDataset(test_data,tokenizer=tokenizer)

In [ ]:
train_dataset[0]

[24361,
 25,
 198,
 36,
 2100,
 4985,
 262,
 1708,
 9546,
 416,
 25449,
 340,
 656,
 262,
 24993,
 1813,
 13,
 198,
 19503,
 521,
 14610,
 1545,
 33706,
 25,
 220,
 198,
 464,
 24993,
 286,
 262,
 1813,
 9546,
 366,
 19503,
 521,
 1,
 318,
 11491,
 11,
 262,
 3376,
 24993,
 318,
 366,
 6726,
 1911]

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=8,shuffle=True,collate_fn=collate_fn,drop_last=True)
val_loader = DataLoader(val_dataset,batch_size=8,shuffle=False,collate_fn=collate_fn,drop_last=True)
test_loader = DataLoader(test_dataset,batch_size=8,shuffle=False,collate_fn=collate_fn,drop_last=True)

In [ ]:
model.to(device)

torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    print("train_loss:"+str(train_loss))
    print("val_loss:"+str(val_loss))

train_loss:24.689333343505858
val_loss:25.331632232666017


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(),lr=0.00005,weight_decay=0.1)

In [ ]:
train_loss,val_loss,_ = trainGpt(model=model,train_loader=train_loader,val_loader=val_loader,optimizer=optimizer,device=device,num_epoch=1,eval_freq=5,eval_iter=5,tokenizer=tokenizer)

Epoch:0 Training_Loss:20.5445125579834 Validation_Loss:20.988650131225587
Epoch:0 Training_Loss:12.654662132263184 Validation_Loss:12.943732070922852
Epoch:0 Training_Loss:9.252968406677246 Validation_Loss:9.594662284851074
Epoch:0 Training_Loss:7.796334838867187 Validation_Loss:7.91939001083374
Epoch:0 Training_Loss:6.788062858581543 Validation_Loss:7.031423091888428
Epoch:0 Training_Loss:6.292921924591065 Validation_Loss:6.405556297302246
Epoch:0 Training_Loss:5.769448852539062 Validation_Loss:6.024568557739258
Epoch:0 Training_Loss:5.407957935333252 Validation_Loss:5.703930854797363
Epoch:0 Training_Loss:5.279924392700195 Validation_Loss:5.477309322357177
Epoch:0 Training_Loss:4.916683769226074 Validation_Loss:5.224872016906739
Epoch:0 Training_Loss:4.699430656433106 Validation_Loss:5.080601406097412
Epoch:0 Training_Loss:4.827888679504395 Validation_Loss:4.9193775177001955
Epoch:0 Training_Loss:4.762573146820069 Validation_Loss:4.785515785217285
Epoch:0 Training_Loss:4.240922737121

In [ ]:
torch.save(model.state_dict(),"model_fine.pth")

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=50256):

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]

        if top_k is not None:

            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            neg_inf = torch.full_like(logits, float("-inf"))
            logits = torch.where(logits < min_val,neg_inf, logits)

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_id is not None and (idx_next == eos_id).all():
          break


        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [ ]:
torch.manual_seed(123)


for entry in test_data[0:1]:

    input_text = formate_input(entry)

    token_ids = generate(
        model=model,
        idx=encode(input_text, tokenizer,device),
        max_new_tokens=50,
        top_k=40,
        temperature=2,
        context_size=cfg["context_length"],
        eos_id=50256
    )
    generated_text = decode(token_ids, tokenizer)
    response_text = (
        generated_text[len(input_text):]
        .replace("\n", "")
        .strip())

    #print(input_text)
    #print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text}")


Model response:
>> 2 the capital to the go goes goes will that she withAnswer:�I it are where you you you I you to where by delicious won on sentence on sentence's you.


In [ ]:
torch.manual_seed(123)
input_text = formate_input(data[5])
token_ids = generate(
        model=model,
        idx=encode(input_text, tokenizer,device),
        max_new_tokens=500,
        context_size=cfg["context_length"],
        eos_id=50256
    )

In [ ]:
text = decode(token_ids,tokenizer)[len(input_text):].strip()

In [ ]:
text

'Answer: \nThe synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for synonym for the synonym for the synonym for for for for for the synonym for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for for f

In [ ]:
import math

def perplexity(loss):
    return math.exp(loss)

print("Train PPL:", perplexity(train_loss[-1]))
print("Val PPL:", perplexity(val_loss[-1]))


Train PPL: 22.958591342098856
Val PPL: 36.26783198363746


In [ ]:
for name, p in model.named_parameters():
    print(f"{name:40s} {p.numel():,}")


word_emb.weight                          51,463,168
pos_emb.weight                           1,048,576
trf.0.norm1.scale                        1,024
trf.0.norm1.shift                        1,024
trf.0.norm1_post.scale                   1,024
trf.0.norm1_post.shift                   1,024
trf.0.norm2.scale                        1,024
trf.0.norm2.shift                        1,024
trf.0.norm2_post.scale                   1,024
trf.0.norm2_post.shift                   1,024
trf.0.M_attn.query.weight                1,048,576
trf.0.M_attn.query.bias                  1,024
trf.0.M_attn.key.weight                  1,048,576
trf.0.M_attn.key.bias                    1,024
trf.0.M_attn.value.weight                1,048,576
trf.0.M_attn.value.bias                  1,024
trf.0.M_attn.proj.weight                 1,048,576
trf.0.M_attn.proj.bias                   1,024
trf.0.ff.layers.0.weight                 4,194,304
trf.0.ff.layers.0.bias                   4,096
trf.0.ff.layers.2.weight       

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")


Total parameters: 406,386,688


In [ ]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(total, trainable)


406386688 406386688


In [ ]:
print(f"{total_params / 1e6:.2f}M")


406.39M
